In [116]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from env_workers import env_worker
import torch.multiprocessing as mp
from utils import get_model
import json
from types import SimpleNamespace
import numpy as np

In [117]:
with open('config.json', "r") as config_file:
    config_file = json.load(config_file)
    config = SimpleNamespace(**config_file)

In [127]:
EPSILON = 0.1
DEVICE = 'cpu'
NUM_WORKERS = 12
STEPS = 1
GAMMA = 1.0
T_MAX = 20
ENTROPY_COEFF = 1e-3
CRITIC_COEFF = 0.5

model = get_model(config.actor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [135]:
parent_remote, worker_remote = zip(*[mp.Pipe() for _ in range(NUM_WORKERS)])
processes = [
    mp.Process(target=env_worker, args=(worker_remote[i], parent_remote[i], i, EPSILON, DEVICE))
    for i in range(NUM_WORKERS)
]

In [ ]:
for p in processes:
    p.start()

Openning Book has been loaded!Openning Book has been loaded!Openning Book has been loaded!Openning Book has been loaded!Openning Book has been loaded!




Openning Book has been loaded!Openning Book has been loaded!

Openning Book has been loaded!
Openning Book has been loaded!
Openning Book has been loaded!
Openning Book has been loaded!
Openning Book has been loaded!


In [137]:
for remote in worker_remote:
    remote.close()

In [131]:
for remote in parent_remote:
    remote.send(('reset', None))
results = [remote.recv() for remote in parent_remote]
states, action_masks = zip(*results)
states = torch.vstack(states)
action_masks = np.vstack(action_masks)

In [132]:
for step in range(100000):
    rewards, state_values, log_probs, entropies, terminates = [], [], [], [], []

    for t in range(T_MAX):
        logits, values = model(states)

        # prepare the mask
        action_masks = torch.tensor(action_masks, dtype=torch.bool)
        action_masks = ~action_masks

        masked_logits = torch.masked_fill(logits, action_masks, value=-torch.inf)
        probs = F.softmax(masked_logits, dim=-1)

        dist = Categorical(probs=probs)
        actions = dist.sample()
        log_prob = dist.log_prob(actions).unsqueeze(1)
        entropy = dist.entropy().unsqueeze(1)

        actions = actions.cpu().detach().tolist()

        for remote, action in zip(parent_remote, actions):
            remote.send(('step', action))


        results = [remote.recv() for remote in parent_remote]
        next_states, action_masks, reward, terminated, outcomes  = zip(*results)
        next_states = torch.vstack(next_states)
        action_masks = np.vstack(action_masks)
        reward = np.vstack(reward)
        terminated = np.vstack(terminated)

        rewards.append(reward)
        state_values.append(values)
        log_probs.append(log_prob)
        entropies.append(entropy)
        terminates.append(terminated)

        states = next_states


    with torch.no_grad():
        _, next_values = model(next_states)

    R = next_values.numpy()
    returns = []
    for i in reversed(range(len(rewards))):
        r = rewards[i]
        term = terminates[i].astype(np.float32)
        R = r + (1 - term) * GAMMA * R
        returns.insert(0, R)

    returns = np.concat(returns, axis=1)
    returns = torch.from_numpy(returns)
    values = torch.concat(state_values, dim=-1)
    log_probs = torch.concat(log_probs, dim=-1)
    entropies = torch.concat(entropies, dim=-1)

    advantage = returns - values
    critic_loss = 0.5 * torch.square(advantage).mean()
    actor_loss = -(advantage.detach() * log_probs).mean() - ENTROPY_COEFF * entropies.mean()

    optimizer.zero_grad()
    model.zero_grad()
    loss = actor_loss + CRITIC_COEFF * critic_loss
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 40)
    optimizer.step()
    print(loss.item())
    print(sum([1 if x == 'win' else 0 for x in outcomes]))
    print('-----')

-1.4209226158624508
0
-----
-1.4132342676767884
0
-----
-1.1201089426521054
0
-----
-1.3834388587316868
0
-----
-1.3637664041293933
0
-----
-1.3834857277697679
0
-----
-1.3617504563947194
0
-----
-1.1951271031847919
0
-----
-1.268312173166805
0
-----
-1.2898973274960586
0
-----
-1.3464579060968724
0
-----
-1.159467043800737
0
-----
-1.236395382772877
0
-----
-1.2519913582650095
0
-----
-1.15236142921281
0
-----
-1.1828212616013214
0
-----
-1.0880671802088093
0
-----
-1.084542122907462
0
-----
-1.162147833244727
0
-----
-1.2115390731936306
0
-----
-1.0808630550650673
0
-----
-1.138136116679022
0
-----
-1.0374873630198578
0
-----
-0.8956737582636851
0
-----
-1.0327580098877784
0
-----
-0.9521664174104499
0
-----
-0.8280504622522264
0
-----
-0.9827562491862517
0
-----
-0.6821857363942769
0
-----
-0.875081048131323
0
-----
-0.9043751688024037
0
-----
-0.9001973043931083
0
-----
-0.8981138224208869
0
-----
-0.8686185547996099
0
-----
-0.7745336360493836
0
-----
-0.7749134718948211
0
-----
-

KeyboardInterrupt: 

In [138]:
results = [remote.send(('reset', None)) for remote in parent_remote]
results = [remote.recv() for remote in parent_remote]
states, action_masks = zip(*results)

In [139]:
states = torch.vstack(states)

In [140]:
torch.nn.functional.softmax(model(states)[0], dim=-1)

tensor([[6.8633e-06, 1.4070e-07, 1.8228e-10, 9.9991e-01, 8.2938e-05, 3.0824e-09,
         4.2687e-10],
        [6.8633e-06, 1.4070e-07, 1.8228e-10, 9.9991e-01, 8.2938e-05, 3.0824e-09,
         4.2687e-10],
        [6.8633e-06, 1.4070e-07, 1.8228e-10, 9.9991e-01, 8.2938e-05, 3.0824e-09,
         4.2687e-10],
        [6.8633e-06, 1.4070e-07, 1.8228e-10, 9.9991e-01, 8.2938e-05, 3.0824e-09,
         4.2687e-10],
        [6.8633e-06, 1.4070e-07, 1.8228e-10, 9.9991e-01, 8.2938e-05, 3.0824e-09,
         4.2687e-10],
        [6.8633e-06, 1.4070e-07, 1.8228e-10, 9.9991e-01, 8.2938e-05, 3.0824e-09,
         4.2687e-10],
        [6.8633e-06, 1.4070e-07, 1.8228e-10, 9.9991e-01, 8.2938e-05, 3.0824e-09,
         4.2687e-10],
        [6.8633e-06, 1.4070e-07, 1.8228e-10, 9.9991e-01, 8.2938e-05, 3.0824e-09,
         4.2687e-10],
        [6.8633e-06, 1.4070e-07, 1.8228e-10, 9.9991e-01, 8.2938e-05, 3.0824e-09,
         4.2687e-10],
        [6.8633e-06, 1.4070e-07, 1.8228e-10, 9.9991e-01, 8.2938e-05, 3.08

In [134]:
torch.save(model, "checkpoint.pt")